In [1]:
# Se deben importar las librerías usadas durante el análisis.

from pathlib import Path
import pandas as pd
import plotly.express as px

In [2]:
# Se debe definir y verificar la ruta del archivo de ventas.

SALES_PATH = Path("../data/sales.csv")
assert SALES_PATH.exists(), f"No existe {SALES_PATH}"

In [3]:
# Se debe cargar y verificar el archivo de ventas.

sales = pd.read_csv(SALES_PATH)
sales.info()
sales.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   OrderID        1000 non-null   int64  
 1   ProductID      1000 non-null   int64  
 2   Category       1000 non-null   object 
 3   ProductName    1000 non-null   object 
 4   Quantity       1000 non-null   int64  
 5   Price          1000 non-null   float64
 6   TotalAmount    1000 non-null   float64
 7   OrderDate      1000 non-null   object 
 8   CustomerID     1000 non-null   int64  
 9   PaymentMethod  1000 non-null   object 
 10  SalesChannel   1000 non-null   object 
 11  IsReturned     1000 non-null   int64  
dtypes: float64(2), int64(5), object(5)
memory usage: 93.9+ KB


,OrderID,ProductID,Category,ProductName,Quantity,Price,TotalAmount,OrderDate,CustomerID,PaymentMethod,SalesChannel,IsReturned
0,1,272,Electronics,Product_272,3,54.21,162.63,2022-04-21 01:48:37,26,Credit Card,In-Store,1
1,2,147,Electronics,Product_147,3,163.68,491.04,2022-02-26 03:09:32,38,Cash,Website,1
2,3,217,Fashion,Product_217,5,205.01,1025.05,2022-07-24 07:00:57,271,Cash,Website,0
3,4,292,Electronics,Product_292,4,163.13,652.52,2022-01-09 09:49:05,414,Cash,Website,1
4,5,423,Fashion,Product_423,3,608.73,1826.19,2022-05-19 09:38:52,368,Cash,Mobile App,1


In [4]:
# Se deben explorar las medidas numéricas de las órdenes de venta.

sales.describe().T

,count,mean,std,min,25%,50%,75%,max
OrderID,1000.0,500.50000,288.819436,1.00,250.7500,500.500,750.2500,1000.00
ProductID,1000.0,302.94600,115.281043,100.00,205.0000,298.500,405.2500,498.00
Quantity,1000.0,3.06900,1.384976,1.00,2.0000,3.000,4.0000,5.00
Price,1000.0,416.30670,223.930219,20.70,234.0175,411.995,612.7125,799.97
TotalAmount,1000.0,1280.74555,942.705635,23.94,488.7500,1068.575,1904.4000,3988.50
CustomerID,1000.0,255.25100,142.823803,1.00,136.7500,260.000,379.2500,500.00
IsReturned,1000.0,0.51600,0.499994,0.00,0.0000,1.000,1.0000,1.00


In [5]:
# Se debe verificar que cada fila es una orden completa y que el valor de venta es consistente.

assert sales.isna().sum().sum() == 0
assert sales["OrderID"].is_unique
assert (
    (sales["Quantity"] * sales["Price"])
    .round(2)
    .eq(sales["TotalAmount"].round(2))
    .all()
)
sales[["OrderID", "Quantity", "Price", "TotalAmount"]].head()

,OrderID,Quantity,Price,TotalAmount
0,1,3,54.21,162.63
1,2,3,163.68,491.04
2,3,5,205.01,1025.05
3,4,4,163.13,652.52
4,5,3,608.73,1826.19


In [6]:
# Se deben preparar la fecha, el valor devuelto, la venta neta y el tipo de día de cada orden.

sales["OrderDate"] = pd.to_datetime(sales["OrderDate"])
sales["OrderMonth"] = sales["OrderDate"].dt.to_period("M").dt.to_timestamp()
sales["DayType"] = sales["OrderDate"].dt.dayofweek.map(
    lambda day: "Fin de semana" if day >= 5 else "Laboral"
)
sales["ReturnedAmount"] = sales["TotalAmount"].where(sales["IsReturned"].eq(1), 0)
sales["NetAmount"] = sales["TotalAmount"] - sales["ReturnedAmount"]

In [7]:
# ¿Cuál es el desempeño comercial global en ventas brutas, devoluciones y ventas netas?

kpi = pd.DataFrame(
    {
        "orders": [sales["OrderID"].nunique()],
        "customers": [sales["CustomerID"].nunique()],
        "gross_sales": [sales["TotalAmount"].sum()],
        "returned_amount": [sales["ReturnedAmount"].sum()],
        "net_sales": [sales["NetAmount"].sum()],
    }
)
kpi["return_rate_by_orders"] = sales["IsReturned"].mean()
kpi["return_rate_by_value"] = kpi["returned_amount"] / kpi["gross_sales"]
kpi["net_sales_per_order"] = kpi["net_sales"] / kpi["orders"]
kpi.round(3)

,orders,customers,gross_sales,returned_amount,net_sales,return_rate_by_orders,return_rate_by_value,net_sales_per_order
0,1000,428,1280745.55,643703.02,637042.53,0.516,0.503,637.043


In [8]:
# ¿Cómo evolucionan las ventas brutas, devoluciones y ventas netas durante el período observado?

monthly_sales = (
    sales.groupby("OrderMonth")[["TotalAmount", "ReturnedAmount", "NetAmount"]]
    .sum()
    .reset_index()
)
monthly_sales = monthly_sales.melt(
    id_vars="OrderMonth", var_name="metric", value_name="amount"
)
fig = px.line(
    monthly_sales,
    x="OrderMonth",
    y="amount",
    color="metric",
    markers=True,
    title="Ventas brutas, devoluciones y ventas netas por mes",
    labels={"OrderMonth": "Mes", "amount": "Valor", "metric": "Métrica"},
)
fig.update_layout(template="plotly_white")
fig.show()

In [9]:
# ¿Qué categorías combinan ventas netas altas con tasas de devolución elevadas?

category_summary = (
    sales.groupby("Category")
    .agg(
        orders=("OrderID", "size"),
        gross_sales=("TotalAmount", "sum"),
        returned_amount=("ReturnedAmount", "sum"),
        net_sales=("NetAmount", "sum"),
        return_rate=("IsReturned", "mean"),
    )
    .reset_index()
)
category_summary = category_summary.sort_values("net_sales", ascending=False)
category_summary.round(3)

,Category,orders,gross_sales,returned_amount,net_sales,return_rate
1,Fashion,343,451214.50,225784.37,225430.13,0.522
0,Electronics,336,440470.24,226316.49,214153.75,0.512
2,Home & Garden,321,389060.81,191602.16,197458.65,0.514


In [10]:
# ¿Cómo se comparan visualmente las ventas netas y las devoluciones entre categorías?

fig = px.bar(
    category_summary,
    x="Category",
    y="net_sales",
    color="return_rate",
    color_continuous_scale="Oranges",
    hover_data=["orders", "gross_sales", "returned_amount"],
    title="Ventas netas y tasa de devolución por categoría",
    labels={"net_sales": "Ventas netas", "return_rate": "Tasa de devolución"},
)
fig.update_coloraxes(colorbar_tickformat=".0%")
fig.update_layout(template="plotly_white")
fig.show()

In [11]:
# ¿Qué clientes aportan mayor valor neto después de considerar sus devoluciones?

customer_summary = (
    sales.groupby("CustomerID")
    .agg(
        orders=("OrderID", "size"),
        gross_sales=("TotalAmount", "sum"),
        returned_amount=("ReturnedAmount", "sum"),
        net_sales=("NetAmount", "sum"),
    )
    .reset_index()
)
customer_summary["return_rate_by_value"] = (
    customer_summary["returned_amount"] / customer_summary["gross_sales"]
)
top_customers = customer_summary.nlargest(10, "net_sales")
top_customers.round(3)

,CustomerID,orders,gross_sales,returned_amount,net_sales,return_rate_by_value
326,383,4,12157.31,3428.45,8728.86,0.282
175,205,4,9065.55,486.40,8579.15,0.054
400,466,8,10623.11,2614.20,8008.91,0.246
122,141,4,10253.65,2824.60,7429.05,0.275
7,9,5,8867.89,2145.66,6722.23,0.242
138,160,5,9912.24,3487.00,6425.24,0.352
135,157,5,11836.95,5532.68,6304.27,0.467
34,40,4,8046.50,1841.68,6204.82,0.229
261,307,3,6136.78,0.00,6136.78,0.000
306,359,2,5710.80,0.00,5710.80,0.000


In [12]:
# ¿Qué productos generan mayor venta neta dentro de cada categoría?

product_summary = (
    sales.groupby(["Category", "ProductName"])
    .agg(
        orders=("OrderID", "size"),
        net_sales=("NetAmount", "sum"),
        return_rate=("IsReturned", "mean"),
    )
    .reset_index()
)
top_products = (
    product_summary.sort_values(["Category", "net_sales"], ascending=[True, False])
    .groupby("Category")
    .head(5)
)
fig = px.bar(
    top_products,
    x="net_sales",
    y="ProductName",
    color="Category",
    facet_col="Category",
    orientation="h",
    hover_data=["orders", "return_rate"],
    title="Cinco productos con mayor venta neta por categoría",
    labels={"net_sales": "Ventas netas", "ProductName": "Producto"},
)
fig.update_layout(template="plotly_white")
fig.show()

In [13]:
# ¿Qué combinaciones de categoría y canal presentan mayor riesgo de devolución?

minimum_orders = 50
return_risk = (
    sales.groupby(["Category", "SalesChannel"])
    .agg(
        orders=("OrderID", "size"),
        gross_sales=("TotalAmount", "sum"),
        returned_amount=("ReturnedAmount", "sum"),
        return_rate=("IsReturned", "mean"),
    )
    .reset_index()
)
return_risk = return_risk[return_risk["orders"] >= minimum_orders]
return_risk = return_risk.sort_values("return_rate", ascending=False)
return_risk.round(3)

,Category,SalesChannel,orders,gross_sales,returned_amount,return_rate
7,Home & Garden,Mobile App,99,115987.45,73969.23,0.576
2,Electronics,Website,107,129756.20,73225.86,0.551
5,Fashion,Website,112,161321.82,83604.72,0.545
8,Home & Garden,Website,104,132400.53,64723.54,0.529
4,Fashion,Mobile App,131,167679.44,87672.87,0.519
1,Electronics,Mobile App,108,152164.20,78806.27,0.500
3,Fashion,In-Store,100,122213.24,54506.78,0.500
0,Electronics,In-Store,121,158549.84,74284.36,0.488
6,Home & Garden,In-Store,118,140672.83,52909.39,0.449


In [14]:
# ¿Dónde se concentra visualmente el riesgo de devolución por categoría y canal?

return_matrix = return_risk.pivot(
    index="Category", columns="SalesChannel", values="return_rate"
)
fig = px.imshow(
    return_matrix,
    color_continuous_scale="Oranges",
    aspect="auto",
    title="Tasa de devolución por categoría y canal",
)
fig.update_layout(template="plotly_white", coloraxis_colorbar_tickformat=".0%")
fig.show()

In [15]:
# ¿Qué segmentos se deben priorizar para investigar y reducir las devoluciones?

priorities = return_risk.copy()
priorities["segment"] = priorities["Category"] + " — " + priorities["SalesChannel"]
priorities = priorities.nlargest(5, "returned_amount")
fig = px.bar(
    priorities.sort_values("returned_amount"),
    x="returned_amount",
    y="segment",
    orientation="h",
    color="return_rate",
    color_continuous_scale="Oranges",
    hover_data=["orders", "gross_sales"],
    title="Segmentos prioritarios por valor devuelto",
    labels={
        "returned_amount": "Valor devuelto",
        "return_rate": "Tasa de devolución",
        "segment": "Categoría — canal",
    },
)
fig.update_coloraxes(colorbar_tickformat=".0%")
fig.update_layout(template="plotly_white")
fig.show()

In [16]:
# ¿El comportamiento comercial cambia entre días laborales y fines de semana?

day_type_summary = (
    sales.groupby("DayType")
    .agg(
        orders=("OrderID", "size"),
        gross_sales=("TotalAmount", "sum"),
        net_sales=("NetAmount", "sum"),
        return_rate=("IsReturned", "mean"),
    )
    .reset_index()
)
fig = px.bar(
    day_type_summary,
    x="DayType",
    y="net_sales",
    color="return_rate",
    color_continuous_scale="Oranges",
    hover_data=["orders", "gross_sales"],
    title="Ventas netas y devoluciones por tipo de día",
    labels={
        "DayType": "Tipo de día",
        "net_sales": "Ventas netas",
        "return_rate": "Tasa de devolución",
    },
)
fig.update_coloraxes(colorbar_tickformat=".0%")
fig.update_layout(template="plotly_white")
fig.show()

In [17]:
# ¿Qué medios de pago combinan ventas netas y tasas de devolución que requieren investigación?

payment_summary = (
    sales.groupby("PaymentMethod")
    .agg(
        orders=("OrderID", "size"),
        gross_sales=("TotalAmount", "sum"),
        returned_amount=("ReturnedAmount", "sum"),
        net_sales=("NetAmount", "sum"),
        return_rate=("IsReturned", "mean"),
    )
    .reset_index()
)
fig = px.bar(
    payment_summary,
    x="PaymentMethod",
    y="net_sales",
    color="return_rate",
    color_continuous_scale="Oranges",
    hover_data=["orders", "returned_amount"],
    title="Ventas netas y devoluciones por medio de pago",
    labels={
        "PaymentMethod": "Medio de pago",
        "net_sales": "Ventas netas",
        "return_rate": "Tasa de devolución",
    },
)
fig.update_coloraxes(colorbar_tickformat=".0%")
fig.update_layout(template="plotly_white")
fig.show()